# 4-Model × 6-Dataset Eval on Colab T4

**One upload button, one download button.**

Runs Qwen1.5B, Qwen-Math-1.5B, Gemma-3-1B, and Qwen-Coder-1.5B
on all 6 eval datasets. Produces 24 xlsx files (4 models × 6 datasets).


In [ ]:
# ── 1. Install ──
!pip install -q llama-cpp-python numpy scipy openpyxl
import os, sys, json, time, re, csv, threading, logging, gc, glob, zipfile, io
from datetime import datetime
print("Ready")


In [ ]:
# ── 2. Define 4 models ──
MODELS = [
    {"label": "qwen-1.5b",
     "url": "https://huggingface.co/Qwen/Qwen2.5-1.5B-Instruct-GGUF/resolve/main/qwen2.5-1.5b-instruct-q4_k_m.gguf",
     "path": "/content/models/qwen2.5-1.5b-instruct-q4_k_m.gguf"},
    {"label": "qwen-math-1.5b",
     "url": "https://huggingface.co/Qwen/Qwen2.5-Math-1.5B-Instruct-GGUF/resolve/main/qwen2.5-math-1.5b-instruct-q4_k_m.gguf",
     "path": "/content/models/qwen2.5-math-1.5b-instruct-q4_k_m.gguf"},
    {"label": "gemma-3-1b",
     "url": "https://huggingface.co/bartowski/gemma-3-1b-it-GGUF/resolve/main/gemma-3-1b-it-Q4_K_M.gguf",
     "path": "/content/models/gemma-3-1b-it-Q4_K_M.gguf"},
    {"label": "qwen-coder-1.5b",
     "url": "https://huggingface.co/Qwen/Qwen2.5-Coder-1.5B-Instruct-GGUF/resolve/main/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf",
     "path": "/content/models/qwen2.5-coder-1.5b-instruct-q4_k_m.gguf"},
]
os.makedirs("/content/models", exist_ok=True)
print(f"4 models defined, ~3.9 GB total download")


## Step 1 — Upload eval files

**Click the button below, select `eval_data.zip` (containing all 6 JSON files), and press Open.**

In [ ]:
# ── 3. Upload eval zip (ONE BUTTON) ──
from google.colab import files
import zipfile

print("Click 'Choose Files' -> select your eval_data.zip with all 6 JSON files")
uploaded = files.upload()

eval_files = []
for fname in uploaded.keys():
    if fname.endswith('.zip'):
        with zipfile.ZipFile(fname, 'r') as zf:
            zf.extractall('/content/eval_data')
        print(f"Extracted {fname}")
        break

eval_files = sorted(glob.glob('/content/eval_data/*.json'))
if not eval_files:
    eval_files = sorted([f for f in uploaded.keys() if f.endswith('.json')])
    os.makedirs('/content/eval_data', exist_ok=True)
    for f in eval_files:
        os.rename(f, f'/content/eval_data/{f}')
    eval_files = sorted(glob.glob('/content/eval_data/*.json'))

print(f"Found {len(eval_files)} eval files:")
for f in eval_files:
    with open(f) as fh:
        d = json.load(fh)
    if isinstance(d, dict) and 'questions' in d:
        d = d['questions']
    print(f"  {os.path.basename(f):35s} {len(d):>5} questions")


## Step 2 — Write pipeline script

In [ ]:
# ── 4. Write pipeline script ──
PIPELINE_SCRIPT = "#!/usr/bin/env python3\n\"\"\"\nStandalone pipeline for Google Colab (T4 GPU).\nPackages the AMD ACT II hackathon pipeline into a single self-contained script.\n\nUsage:\n    python3 colab_pipeline.py --eval data/eval/training-v1.json --model /path/to/model.gguf\n    python3 colab_pipeline.py --eval data/eval/validation-v1.json --model /path/to/model.gguf --gpu-layers -1\n\nOutputs:\n    results/colab_run_TIMESTAMP.json       Full per-question results\n    results/colab_run_TIMESTAMP_summary.txt Human-readable summary\n\"\"\"\n\nimport argparse\nimport csv\nimport hashlib\nimport json\nimport logging\nimport os\nimport re\nimport sys\nimport threading\nimport time\nfrom collections import Counter\nfrom dataclasses import dataclass, field\nfrom datetime import datetime\nfrom typing import Any, Dict, List, Optional, Tuple\n\nlogging.basicConfig(\n    level=logging.INFO,\n    format=\"%(asctime)s [%(levelname)s] %(message)s\",\n    stream=sys.stderr,\n)\nlogger = logging.getLogger(\"colab_pipeline\")\n\n# =========================================================================\n# CONFIG\n# =========================================================================\n\nDEFAULT_MODEL = \"/models/qwen2.5-1.5b-instruct-q4_k_m.gguf\"\nN_GPU_LAYERS = -1\nN_CTX = 2048\nN_THREADS = 4\nMAX_TOKENS = 256\nCOMPLEXITY_SIMPLE_MAX = 0.3\nCONSENSUS_SAMPLES = 1\n\nDETERMINISTIC_CATEGORIES = {\"math\", \"logic\", \"sentiment\", \"ner\", \"factual\", \"code_debug\"}\nNAKED_CATEGORIES = {\"ner\", \"summarization\", \"factual\", \"logic\", \"math\"}\n\nCATEGORIES_8WAY = [\n    \"code_debug\", \"code_gen\", \"factual\", \"logic\",\n    \"math\", \"ner\", \"sentiment\", \"summarization\",\n]\n\nPRIORITY = {\n    \"code_debug\": 8, \"code_gen\": 7, \"math\": 6, \"logic\": 5,\n    \"sentiment\": 4, \"ner\": 3, \"summarization\": 2, \"factual\": 1,\n}\n\nDET_CATEGORY_MAP = {\n    \"math\": \"math_arithmetic\",\n    \"logic\": \"logical_reasoning\",\n    \"sentiment\": \"sentiment\",\n    \"ner\": \"named_entity_recognition\",\n    \"factual\": \"other_complex\",\n    \"code_debug\": \"code_debugging\",\n    \"code_gen\": \"code_debugging\",\n    \"summarization\": \"summarization\",\n    \"general\": \"other_complex\",\n}\n\nHUMAN_NAMES = {\n    \"code_debug\": \"Code Debugging\", \"code_gen\": \"Code Generation\",\n    \"factual\": \"Factual Knowledge\", \"logic\": \"Logical Reasoning\",\n    \"math\": \"Math Reasoning\", \"ner\": \"Named Entity Recognition\",\n    \"sentiment\": \"Sentiment\", \"summarization\": \"Summarisation\",\n}\n\n\n# =========================================================================\n# DETERMINISTIC SOLVERS\n# =========================================================================\n\ndef solve_arithmetic(prompt: str, category: str = \"\") -> Optional[str]:\n    \"\"\"Solve arithmetic expressions using safe eval.\"\"\"\n    prompt_clean = prompt.strip()\n    # Pure arithmetic: numbers and operators only\n    arith_pattern = re.compile(\n        r'^[\\d\\s\\+\\-\\*\\/\\(\\)\\%\\.\\,]+$'\n    )\n    # Check if it's a calculation request\n    calc_keywords = re.compile(\n        r'(calculate|compute|what\\s+is|result\\s+of|solve|evaluate|simplify)\\b',\n        re.IGNORECASE\n    )\n    if not calc_keywords.search(prompt_clean) and not arith_pattern.match(prompt_clean):\n        return None\n\n    # Extract the expression\n    nums = re.findall(r'[\\d]+(?:\\s*[\\+\\-\\*\\/\\(\\)]\\s*[\\d]+)+', prompt_clean)\n    if not nums:\n        nums = re.findall(r'[\\d]+(?:\\s*[\\+\\-\\*\\/\\(\\)]\\s*[\\d]+)*\\s*=', prompt_clean)\n    if not nums:\n        nums = re.findall(r'[\\d\\s\\+\\-\\*\\/\\(\\)\\%]+', prompt_clean)\n    if not nums:\n        return None\n\n    expr = nums[0].strip().rstrip('=')\n    try:\n        # Safe eval - only allow basic math\n        result = eval(expr, {\"__builtins__\": {}}, {})\n        if isinstance(result, (int, float)):\n            return f\"{result:.4f}\".rstrip('0').rstrip('.') if isinstance(result, float) else str(result)\n    except Exception:\n        pass\n    return None\n\n\ndef solve_logic(prompt: str, category: str = \"\") -> Optional[str]:\n    \"\"\"Solve simple logic puzzles (truth-tellers, liars, syllogisms).\"\"\"\n    # Truth-teller / liar patterns\n    truth_teller = re.compile(\n        r'(truth.?teller|liar|always\\s+tells?\\s+(the\\s+)?truth|always\\s+lies)',\n        re.IGNORECASE\n    )\n    if not truth_teller.search(prompt):\n        return None\n    return None  # Too complex for deterministic - let LLM handle\n\n\ndef solve_sentiment(prompt: str, category: str = \"\") -> Optional[str]:\n    \"\"\"Simple sentiment detection for clear cases.\"\"\"\n    prompt_lower = prompt.lower()\n    sentiment_words = {\n        'positive': {'happy', 'great', 'excellent', 'wonderful', 'amazing', 'fantastic',\n                     'love', 'beautiful', 'perfect', 'brilliant', 'delighted', 'pleased'},\n        'negative': {'terrible', 'awful', 'horrible', 'hate', 'bad', 'worst',\n                     'disappointed', 'sad', 'angry', 'frustrating', 'poor', 'dreadful'},\n        'neutral': {'the', 'this', 'that', 'it', 'is', 'was', 'are', 'were', 'has', 'have',\n                    'said', 'says', 'according', 'reported'}\n    }\n    # Count sentiment signals\n    pos = sum(1 for w in sentiment_words['positive'] if w in prompt_lower)\n    neg = sum(1 for w in sentiment_words['negative'] if w in prompt_lower)\n\n    total = pos + neg\n    if total >= 3:\n        ratio = pos / total\n        if ratio > 0.66:\n            return \"positive\"\n        elif ratio < 0.33:\n            return \"negative\"\n    return None\n\n\ndef solve_ner(prompt: str, category: str = \"\") -> Optional[str]:\n    \"\"\"Simple NER for common patterns.\"\"\"\n    entities = []\n    # Person names (Captain Formals pattern)\n    persons = re.findall(r'(?:Dr\\.|Mr\\.|Mrs\\.|Ms\\.|Prof\\.)\\s+[A-Z][a-z]+(?:\\s+[A-Z][a-z]+)*', prompt)\n    entities.extend(persons)\n\n    # Organizations\n    orgs = re.findall(r'\\b(?:[A-Z][a-z]+(?:\\s+[A-Z][a-z]+)*\\s+(?:Inc|Corp|LLC|Ltd|Group|Technologies|Systems|Pharmaceuticals|Laboratories|University|Institute|Hospital|Airlines))\\b', prompt)\n    entities.extend(orgs)\n\n    # Dates\n    dates = re.findall(r'\\b(?:January|February|March|April|May|June|July|August|September|October|November|December)\\s+\\d{1,2},?\\s+\\d{4}\\b', prompt)\n    entities.extend(dates)\n\n    if entities:\n        return \", \".join(entities)\n    return None\n\n\ndef solve_factual_qa(prompt: str, category: str = \"\") -> Optional[str]:\n    \"\"\"Simple factual QA for common knowledge.\"\"\"\n    prompt_lower = prompt.lower().strip()\n\n    # Capital city questions\n    capital_q = re.compile(r'(?:what\\s+is\\s+the\\s+capital|capital\\s+of)\\s+(\\w+(?:\\s+\\w+)?)\\s*\\?', re.IGNORECASE)\n    m = capital_q.search(prompt)\n    if m:\n        country = m.group(1).strip().lower()\n        capitals = {\n            \"france\": \"Paris\", \"germany\": \"Berlin\", \"italy\": \"Rome\",\n            \"spain\": \"Madrid\", \"uk\": \"London\", \"united kingdom\": \"London\",\n            \"japan\": \"Tokyo\", \"china\": \"Beijing\", \"india\": \"New Delhi\",\n            \"australia\": \"Canberra\", \"canada\": \"Ottawa\", \"brazil\": \"Bras\u00edlia\",\n            \"russia\": \"Moscow\", \"usa\": \"Washington, D.C.\", \"united states\": \"Washington, D.C.\",\n            \"united states of america\": \"Washington, D.C.\", \"egypt\": \"Cairo\",\n            \"nigeria\": \"Abuja\", \"kenya\": \"Nairobi\", \"south africa\": \"Pretoria\",\n        }\n        if country in capitals:\n            return capitals[country]\n\n    return None\n\n\ndef solve_code_debugging(prompt: str, category: str = \"\") -> Optional[str]:\n    \"\"\"Fix common bug patterns in code.\"\"\"\n    return None  # Too complex to be reliable\n\n# Map deterministic solver names to functions\nDET_SOLVER_FNS = {\n    \"math_arithmetic\": solve_arithmetic,\n    \"logical_reasoning\": solve_logic,\n    \"sentiment\": solve_sentiment,\n    \"named_entity_recognition\": solve_ner,\n    \"other_complex\": solve_factual_qa,\n    \"code_debugging\": solve_code_debugging,\n}\n\n\n# =========================================================================\n# 8-WAY CATEGORY CLASSIFIER (regex-based)\n# =========================================================================\n\ndef _score_code_debug(prompt: str) -> float:\n    \"\"\"Score for code_debugging category.\"\"\"\n    score = 0.0\n    if re.search(r'\\b(bug|error|fix|debug|issue|crash|broken|incorrect|wrong|fail|not work|debugging)\\b',\n                 prompt, re.IGNORECASE):\n        score += 3.0\n    if re.search(r'\\b(exception|traceback|typeerror|valueerror|keyerror|indexerror|attributeerror|importerror)\\b',\n                 prompt, re.IGNORECASE):\n        score += 2.0\n    if re.search(r'\\b(def\\s+\\w+\\s*\\(|class\\s+\\w+|import\\s+\\w+|return\\s+|print\\s*\\()', prompt):\n        score += 2.0\n    if re.search(r'\\b(test|output|result|expected|actual)\\b.*\\b(different|mismatch|wrong|incorrect)\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.5\n    if re.search(r'\\b(bug\\s+in\\s+the\\s+(Python|code|function)|the\\s+bug\\sin)\\b', prompt, re.IGNORECASE):\n        score += 1.0\n    # Large code block\n    code_lines = len(re.findall(r'^\\s*(def |class |import |from |return |if |for |while |with |try:)', prompt, re.MULTILINE))\n    score += min(code_lines * 0.5, 2.0)\n    return score\n\n\ndef _score_code_gen(prompt: str) -> float:\n    \"\"\"Score for code_generation category.\"\"\"\n    score = 0.0\n    if re.search(r'\\b(write|create|implement|generate|build|code|program|function|method|class)\\b',\n                 prompt, re.IGNORECASE):\n        score += 2.0\n    if re.search(r'\\b(Write\\s+a\\s+(Python|Java|JavaScript|C\\+\\+|function)|Implement\\s+(a\\s+)?(function|class|method|algorithm))\\b',\n                 prompt):\n        score += 2.0\n    if re.search(r'\\b(return|print|input|def |class |import )', prompt):\n        score += 1.5\n    if re.search(r'\\b(Write\\s+a\\s+function|write\\s+the\\s+code|generate\\s+code|implement\\s+(a|the)\\s+function)\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.0\n    if re.search(r'\\b(using\\s+(Python|recursion|iteration|list\\s+comprehension|dictionary|set))\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.0\n    return score\n\n\ndef _score_factual(prompt: str) -> float:\n    \"\"\"Score for factual_knowledge category.\"\"\"\n    score = 0.0\n    if re.search(r'\\b(what|who|when|where|why|which|how)\\b.*\\?', prompt):\n        score += 2.0\n    if re.search(r'\\b(known\\s+as|called|also\\s+known\\s+as|referred\\s+to\\s+as)\\b', prompt, re.IGNORECASE):\n        score += 1.5\n    if re.search(r'\\b(fact|information|knowledge|defined\\s+as|meaning|definition|explain|describe)\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.5\n    if re.search(r'\\b(country|capital|city|river|mountain|ocean|planet|star|element|chemical)\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.0\n    if re.search(r'\\b(history|discovered|invented|founded|born|died|president|king|queen|leader)\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.0\n    if re.search(r'\\b(What\\s+is|Who\\s+(is|was)|When\\s+(did|was)|Where\\s+(is|was)|How\\s+(many|much|far|long|old))\\b',\n                 prompt):\n        score += 1.0\n    return score\n\n\ndef _score_logic(prompt: str) -> float:\n    \"\"\"Score for logical_reasoning category.\"\"\"\n    score = 0.0\n    if re.search(r'\\b(if\\s+.*\\bthen|either\\s+.*\\bor|neither\\s+.*\\bnor|all\\s+.*\\bare|no\\s+.*\\bare)\\b',\n                 prompt, re.IGNORECASE):\n        score += 2.0\n    if re.search(r'\\b(logic|reasoning|conclusion|infer|deduce|premise|assumption|implication)\\b',\n                 prompt, re.IGNORECASE):\n        score += 2.0\n    if re.search(r'\\b(must\\s+be|cannot\\s+be|could\\s+be|always|never|sometimes|necessarily|possibly)\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.5\n    if re.search(r'\\b(truth.?teller|liar|knights?|knaves?|puzzle|riddle)\\b', prompt, re.IGNORECASE):\n        score += 2.0\n    if re.search(r'\\b(statement|contradiction|paradox|syllogism|valid|invalid)\\b', prompt, re.IGNORECASE):\n        score += 2.0\n    if re.search(r'\\b(which\\s+of\\s+the\\s+following|choose|select|pick)\\b', prompt, re.IGNORECASE):\n        score += 1.0\n    if re.search(r'\\b(only|every|some|none|all|no\\s+one|everyone)\\b.*\\b(is|are|can|cannot|will|must)\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.0\n    return score\n\n\ndef _score_math(prompt: str) -> float:\n    \"\"\"Score for math_reasoning category.\"\"\"\n    score = 0.0\n    # Numbers and math operators\n    if re.search(r'[\\d]+[\\s]*[\\+\\-\\*\\/\\^\\%][\\s]*[\\d]+', prompt):\n        score += 2.0\n    if re.search(r'\\b(calculate|compute|solve|evaluate|simplify|find|determine)\\b.*[\\d]', prompt, re.IGNORECASE):\n        score += 2.0\n    if re.search(r'\\b(divided\\s+by|multiplied\\s+by|plus|minus|times|over|percent|percentage|fraction|decimal)\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.5\n    if re.search(r'\\b(distance|speed|rate|time|volume|area|length|width|height|radius|diameter)\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.0\n    if re.search(r'\\b(equation|formula|theorem|proof|derivative|integral|sum|product|function)\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.5\n    if re.search(r'\\b(probability|permutation|combination|mean|median|mode|standard\\s+deviation|variance)\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.5\n    if re.search(r'\\b[A-Za-z]\\s*=\\s*[\\d]', prompt):\n        score += 1.0\n    if re.search(r'\\b(what\\s+is\\s+the\\s+(value|result|sum|product|difference|average|total))\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.0\n    return score\n\n\ndef _score_ner(prompt: str) -> float:\n    \"\"\"Score for named_entity_recognition category.\"\"\"\n    score = 0.0\n    # Named entities (Capitalized words)\n    capitalized = len(re.findall(r'\\b[A-Z][a-z]+\\b', prompt))\n    if capitalized > 5:\n        score += 2.0\n    if re.search(r'\\b(identify|find|extract|list|locate|tag|label|recognize)\\b.*\\b(entity|name|person|organization|location|date|place)\\b',\n                 prompt, re.IGNORECASE):\n        score += 2.5\n    if re.search(r'\\b(NER|named\\s+entity|entity\\s+recognition|entity\\s+extraction|entity\\s+tagging)\\b',\n                 prompt, re.IGNORECASE):\n        score += 3.0\n    if re.search(r'\\b(Dr\\.|Mr\\.|Mrs\\.|Ms\\.|Prof\\.|CEO|President|Director|Professor|Doctor)\\b', prompt):\n        score += 1.5\n    if re.search(r'\\b(Inc|Corp|LLC|Ltd|University|Hospital|Foundation|Organization|Agency|Department)\\b',\n                 prompt):\n        score += 1.0\n    if re.search(r'\\b(January|February|March|April|May|June|July|August|September|October|November|December)\\s+\\d{1,2}',\n                 prompt):\n        score += 1.0\n    if re.search(r'\\b(patient|diagnosis|symptom|treatment|disease|syndrome|condition)\\b', prompt, re.IGNORECASE):\n        score += 1.0\n    return score\n\n\ndef _score_sentiment(prompt: str) -> float:\n    \"\"\"Score for sentiment_classification category.\"\"\"\n    score = 0.0\n    positive_words = re.findall(r'\\b(good|great|excellent|amazing|fantastic|wonderful|happy|love|beautiful|perfect|brilliant|delighted|pleased|awesome|positive)\\b', prompt, re.IGNORECASE)\n    negative_words = re.findall(r'\\b(bad|terrible|awful|horrible|hate|worst|disappointed|sad|angry|frustrating|poor|dreadful|negative|unpleasant|disgusting)\\b', prompt, re.IGNORECASE)\n    pos_count = len(positive_words)\n    neg_count = len(negative_words)\n\n    if re.search(r'\\b(sentiment|emotion|feeling|opinion|mood|reaction|attitude)\\b', prompt, re.IGNORECASE):\n        score += 2.0\n    if re.search(r'\\b(classify|determine|identify|analyze|detect|assess|evaluate)\\b.*\\b(sentiment|emotion|feeling|opinion)\\b',\n                 prompt, re.IGNORECASE):\n        score += 2.0\n    if re.search(r'\\b(positive|negative|neutral)\\b', prompt, re.IGNORECASE):\n        score += 1.5\n    score += min(pos_count * 0.5, 1.5)\n    score += min(neg_count * 0.5, 1.5)\n    if re.search(r'\\b(movie|review|product|service|customer|experience|feedback|comment)\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.0\n    return score\n\n\ndef _score_summarization(prompt: str) -> float:\n    \"\"\"Score for summarisation category.\"\"\"\n    score = 0.0\n    if re.search(r'\\b(summarize|summarise|summary|brief|condense|shorten|synopsis|abstract|overview|gist|tl;dr|tl dr)\\b',\n                 prompt, re.IGNORECASE):\n        score += 3.0\n    if len(prompt.split()) > 100:\n        score += 2.0\n    if re.search(r'\\b(in\\s+one\\s+sentence|in\\s+a\\s+few\\s+sentences|in\\s+your\\s+own\\s+words|in\\s+short|concisely|briefly)\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.5\n    if re.search(r'\\b(article|text|passage|paragraph|document|report|paper|chapter|section|email|letter|story)\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.0\n    if re.search(r'\\b(extract|key\\s+points|main\\s+idea|main\\s+points|essential|core)\\b',\n                 prompt, re.IGNORECASE):\n        score += 1.0\n    # Long text indicates need for summarization\n    word_count = len(prompt.split())\n    if word_count > 150:\n        score += 1.0\n    return score\n\n\nSCORERS: Dict[str, Any] = {\n    \"code_debug\": _score_code_debug,\n    \"code_gen\": _score_code_gen,\n    \"factual\": _score_factual,\n    \"logic\": _score_logic,\n    \"math\": _score_math,\n    \"ner\": _score_ner,\n    \"sentiment\": _score_sentiment,\n    \"summarization\": _score_summarization,\n}\n\n\ndef classify(prompt: str) -> Tuple[str, float, Dict[str, float]]:\n    \"\"\"8-way category classifier. Returns (category, confidence, all_scores).\"\"\"\n    scores: Dict[str, float] = {}\n    for cat, scorer in SCORERS.items():\n        scores[cat] = scorer(prompt)\n    # Highest score wins, ties broken by priority\n    best_cat = max(scores, key=lambda c: (scores[c], PRIORITY.get(c, 0)))\n    second_best = sorted(scores.items(), key=lambda x: (-x[1], -PRIORITY.get(x[0], 0)))[1]\n    confidence = scores[best_cat] - second_best[1]\n    return best_cat, confidence, scores\n\n\n# =========================================================================\n# COMPLEXITY SCORER\n# =========================================================================\n\ndef score_complexity(prompt: str, category: str = \"factual\") -> float:\n    \"\"\"Per-category complexity score (0.0\u20131.0).\"\"\"\n    if category == \"code_gen\":\n        return _complexity_code_gen(prompt)\n    elif category == \"code_debug\":\n        return _complexity_code_debug(prompt)\n    elif category == \"math\":\n        return _complexity_math(prompt)\n    elif category == \"logic\":\n        return _complexity_logic(prompt)\n    elif category == \"factual\":\n        return _complexity_factual(prompt)\n    elif category == \"sentiment\":\n        return _complexity_sentiment(prompt)\n    elif category == \"ner\":\n        return _complexity_ner(prompt)\n    elif category == \"summarization\":\n        return _complexity_summarization(prompt)\n    return 0.5\n\n\ndef _complexity_code_gen(prompt: str) -> float:\n    signals = 0.0\n    func_count = len(re.findall(r'\\b(def\\s+\\w+\\s*\\(|function\\s+\\w+|class\\s+\\w+)', prompt))\n    loop_depth = len(re.findall(r'\\b(for|while)\\s+\\w', prompt))\n    import_count = len(re.findall(r'\\b(import|from)\\s+\\w+', prompt))\n    code_len = len(re.findall(r'^.*$', prompt, re.MULTILINE))\n    signals += min(func_count * 0.15, 0.3)\n    signals += min(loop_depth * 0.1, 0.2)\n    signals += min(import_count * 0.1, 0.2)\n    signals += min(code_len / 500, 0.3)\n    return min(signals, 1.0)\n\n\ndef _complexity_code_debug(prompt: str) -> float:\n    signals = 0.0\n    error_count = len(re.findall(r'\\b(error|bug|exception|issue|wrong|incorrect|fail)', prompt, re.IGNORECASE))\n    code_size = len(re.findall(r'^.*$', prompt, re.MULTILINE))\n    signals += min(error_count * 0.15, 0.4)\n    signals += min(code_size / 400, 0.4)\n    sub_funcs = len(re.findall(r'\\b(def\\s+\\w+\\s*\\()', prompt))\n    signals += min(sub_funcs * 0.1, 0.2)\n    return min(signals, 1.0)\n\n\ndef _complexity_math(prompt: str) -> float:\n    signals = 0.0\n    ops = len(re.findall(r'[\\+\\-\\*\\/\\^\\%]', prompt))\n    paren_depth = max(len(re.findall(r'\\(', prompt)), len(re.findall(r'\\)', prompt)))\n    proof = 1 if re.search(r'\\b(proof|theorem|lemma|axiom|prove|show\\s+that)\\b', prompt, re.IGNORECASE) else 0\n    vars = len(set(re.findall(r'\\b[A-Za-z]\\b', prompt)))\n    signals += min(ops * 0.05, 0.2)\n    signals += min(paren_depth * 0.05, 0.2) if paren_depth > 3 else 0\n    signals += proof * 0.3\n    signals += min(vars * 0.02, 0.3)\n    return min(signals, 1.0)\n\n\ndef _complexity_logic(prompt: str) -> float:\n    signals = 0.0\n    conditions = len(re.findall(r'\\b(if|when|unless|provided|given\\s+that)\\b', prompt, re.IGNORECASE))\n    negations = len(re.findall(r'\\b(not|no|none|never|neither|nor|without)\\b', prompt, re.IGNORECASE))\n    quantifiers = len(re.findall(r'\\b(all|every|some|any|none|no\\s+one|each|both|either|neither|most)\\b', prompt, re.IGNORECASE))\n    signals += min(conditions * 0.1, 0.3)\n    signals += min(negations * 0.1, 0.2)\n    signals += min(quantifiers * 0.1, 0.2)\n    puzzle = 1 if re.search(r'\\b(knights?|knaves?|truth.?teller|liar|puzzle)', prompt, re.IGNORECASE) else 0\n    signals += puzzle * 0.3\n    return min(signals, 1.0)\n\n\ndef _complexity_factual(prompt: str) -> float:\n    signals = 0.0\n    specificity = len(re.findall(r'\\b(January|February|March|April|May|June|July|August|September|October|November|December|\\d{4}|19\\d{2}|20\\d{2})\\b', prompt))\n    multi_hop = len(re.findall(r'\\b(because|therefore|however|although|despite|while|since|as\\s+a\\s+result)\\b', prompt, re.IGNORECASE))\n    ambiguity = len(re.findall(r'\\b(might|maybe|possibly|probably|likely|unclear|unknown|uncertain|ambiguous|controversial)\\b', prompt, re.IGNORECASE))\n    signals += min(specificity * 0.1, 0.3)\n    signals += min(multi_hop * 0.15, 0.3)\n    signals += min(ambiguity * 0.15, 0.2)\n    word_count = len(prompt.split())\n    signals += min(word_count / 200, 0.2)\n    return min(signals, 1.0)\n\n\ndef _complexity_sentiment(prompt: str) -> float:\n    signals = 0.0\n    text_len = len(prompt.split())\n    signals += min(text_len / 200, 0.4)\n    subtle = 1 if re.search(r'\\b(sarcasm|irony|hedging|subtle|nuance|mixed|conflicting|contradictory)\\b', prompt, re.IGNORECASE) else 0\n    signals += subtle * 0.3\n    entities = len(re.findall(r'\\b[A-Z][a-z]+(?:\\s+[A-Z][a-z]+)*\\b', prompt))\n    if entities > 3:\n        signals += 0.2\n    explicit_marker = 1 if re.search(r'\\b(classify|determine|identify)\\b.*\\b(sentiment|emotion|feeling)\\b', prompt, re.IGNORECASE) else 0\n    if explicit_marker:\n        signals -= 0.1\n    return max(0.0, min(signals, 1.0))\n\n\ndef _complexity_ner(prompt: str) -> float:\n    signals = 0.0\n    entity_density = len(re.findall(r'\\b[A-Z][a-z]+\\b', prompt)) / max(len(prompt.split()), 1)\n    signals += min(entity_density * 2.0, 0.4)\n    text_len = len(prompt.split())\n    signals += min(text_len / 200, 0.3)\n    domain = 1 if re.search(r'\\b(biomedical|financial|legal|clinical|technical|scientific|medical)\\b', prompt, re.IGNORECASE) else 0\n    signals += domain * 0.3\n    return min(signals, 1.0)\n\n\ndef _complexity_summarization(prompt: str) -> float:\n    signals = 0.0\n    word_count = len(prompt.split())\n    signals += min(word_count / 500, 0.4)\n    multi_source = len(re.findall(r'\\b(however|but|whereas|although|on\\s+the\\s+(other\\s+)?hand|in\\s+contrast|conversely)\\b', prompt, re.IGNORECASE))\n    signals += min(multi_source * 0.1, 0.3)\n    bullet = 1 if re.search(r'\\b(bullet|point|list|itemize|enumerate|numbered)\\b', prompt, re.IGNORECASE) else 0\n    signals += bullet * 0.2\n    return min(signals, 1.0)\n\n\n# =========================================================================\n# DYNAMIC PROMPTS\n# =========================================================================\n\n_ANTI_PREAMBLE = (\n    \" English only. Start with the answer directly \u2014 no greeting, \"\n    \"no 'I will', no 'The user asks', no meta-commentary. \"\n    \"First word = the answer.\"\n)\n\n_CATEGORY_PROMPTS: dict[str, dict[str, str]] = {\n    \"code_gen\": {\n        \"low\": \"You are a code generation assistant. Generate clean, correct code. Answer with just the code.\",\n        \"medium\": \"You are a code generation assistant. Generate working, well-structured code.\",\n        \"high\": \"You are a code generation assistant. Generate correct, efficient code with proper edge case handling.\",\n    },\n    \"code_debug\": {\n        \"low\": \"You are a code debugging assistant. Identify and fix the bug. Answer with the corrected code.\",\n        \"medium\": \"You are a code debugging assistant. Identify the bug and provide the corrected version.\",\n        \"high\": \"You are a code debugging assistant. Find all bugs, explain the fix, provide corrected code.\",\n    },\n    \"math\": {\n        \"low\": \"You are a math assistant. Solve the problem step by step. Answer with the final result.\",\n        \"medium\": \"You are a math assistant. Solve with clear reasoning. End with the answer.\",\n        \"high\": \"You are a math assistant. Work through the problem systematically. Final answer on its own line.\",\n    },\n    \"logic\": {\n        \"low\": \"You are a logic reasoning assistant. Determine the correct answer concisely.\",\n        \"medium\": \"You are a logic reasoning assistant. Think step by step and state your conclusion.\",\n        \"high\": \"You are a logic reasoning assistant. Reason systematically and provide the final answer.\",\n    },\n    \"factual\": {\n        \"low\": \"You are a factual knowledge assistant. Answer accurately and concisely.\",\n        \"medium\": \"You are a factual knowledge assistant. Provide accurate, well-sourced information.\",\n        \"high\": \"You are a factual knowledge assistant. Give a comprehensive, accurate answer.\",\n    },\n    \"sentiment\": {\n        \"low\": \"You are a sentiment classifier. Respond with exactly: positive, negative, or neutral.\",\n        \"medium\": \"You are a sentiment classifier. Analyze the sentiment and respond with one word: positive, negative, or neutral.\",\n        \"high\": \"You are a sentiment analyst. Consider tone, context, and nuance. Respond with one label: positive, negative, or neutral.\",\n    },\n    \"ner\": {\n        \"low\": \"You are an NER system. List all named entities found in the text.\",\n        \"medium\": \"You are an NER system. Extract all named entities: persons, organizations, locations, dates.\",\n        \"high\": \"You are an NER system. Extract and label all named entities. Format: Type: Entity.\",\n    },\n    \"summarization\": {\n        \"low\": \"You are a summarization assistant. Summarize the text concisely.\",\n        \"medium\": \"You are a summarization assistant. Provide a clear summary covering key points.\",\n        \"high\": \"You are a summarization assistant. Provide a comprehensive summary preserving all key information.\",\n    },\n}\n\n\ndef build_system_prompt(category: str, complexity_score: float = 0.5,\n                        custom_instructions: str = \"\") -> str:\n    \"\"\"Build a category-and-complexity-aware system prompt.\"\"\"\n    tier = \"low\" if complexity_score < 0.3 else (\"medium\" if complexity_score < 0.6 else \"high\")\n    prompts = _CATEGORY_PROMPTS.get(category, _CATEGORY_PROMPTS.get(\"factual\"))\n    base = prompts.get(tier, prompts.get(\"medium\", \"\"))\n    result = base + _ANTI_PREAMBLE\n    if custom_instructions:\n        result += \" \" + custom_instructions\n    return result\n\n\ndef get_max_tokens(category: str, complexity_score: float = 0.5) -> int:\n    \"\"\"Per-category max generation tokens.\"\"\"\n    tok_map = {\n        \"code_gen\": 250, \"code_debug\": 200, \"math\": 200,\n        \"logic\": 200, \"factual\": 200, \"sentiment\": 60,\n        \"ner\": 120, \"summarization\": 120,\n    }\n    base = tok_map.get(category, 150)\n    if complexity_score > 0.6:\n        return min(int(base * 1.5), 400)\n    return base\n\n\nNER_ONE_SHOT_EXAMPLE = (\n    \"Example input: 'Dr. John Smith from Pfizer met with Prof. Jane Doe at Stanford University on March 15, 2024.'\\n\"\n    \"Example output: Person: Dr. John Smith, Prof. Jane Doe | Organization: Pfizer, Stanford University | Date: March 15, 2024\"\n)\n\n\ndef build_merged_prompt(primary_category: str, secondary_category: str,\n                        complexity_score: float = 0.5,\n                        custom_instructions: str = \"\") -> str:\n    \"\"\"Build prompt when classifier has low confidence between 2 categories.\"\"\"\n    primary = _CATEGORY_PROMPTS.get(primary_category, {}).get(\"medium\", \"\")\n    secondary = _CATEGORY_PROMPTS.get(secondary_category, {}).get(\"medium\", \"\")\n    base = (\n        f\"This task could be either {HUMAN_NAMES.get(primary_category, primary_category)} \"\n        f\"or {HUMAN_NAMES.get(secondary_category, secondary_category)}. \"\n        f\"{primary} If it matches {HUMAN_NAMES.get(secondary_category, secondary_category)} instead: {secondary}\"\n    )\n    return base + _ANTI_PREAMBLE\n\n\n# =========================================================================\n# QUALITY CONFIG\n# =========================================================================\n\nQC_CONFIG = {\n    \"code_debug\": {\"metric\": \"top3_gap\", \"threshold\": 2.6667},\n    \"code_gen\": {\"metric\": \"margin\", \"threshold\": 0.6},\n    \"factual\": {\"metric\": \"inverse_active\", \"threshold\": 1.0},\n    \"logic\": {\"metric\": \"top3_gap\", \"threshold\": 2.6667},\n    \"math\": {\"metric\": \"max_score\", \"threshold\": 2.0},\n    \"ner\": {\"metric\": \"top_over_avg\", \"threshold\": 4.4444},\n    \"sentiment\": {\"metric\": \"top3_gap\", \"threshold\": 2.6667},\n    \"summarization\": {\"metric\": \"top3_gap\", \"threshold\": 2.6667},\n}\nQC_POLICY = {\"on_fail\": \"escalate\", \"min_accept_precision\": 0.85}\n\n\n# =========================================================================\n# STAGE 0: PRE-FILTER\n# =========================================================================\n\n@dataclass\nclass Stage0Result:\n    action: str = \"continue\"\n    direct_answer: Optional[str] = None\n    category: Optional[str] = None\n    flags: Dict[str, bool] = field(default_factory=dict)\n\n\nRE_GREETING = re.compile(r\"^(hi|hello|hey|thanks|thank you|ok|okay)[\\s!.]*$\", re.I)\nRE_MATH_OPERATION = re.compile(\n    r'^[\\d\\s\\+\\-\\*\\/\\(\\)\\%\\.\\,]+$'\n)\n\n\ndef stage0(prompt: str) -> Stage0Result:\n    \"\"\"Pre-filter: bypass trivial cases.\"\"\"\n    cleaned = prompt.strip()\n\n    # Tier 0A: Greetings\n    if RE_GREETING.match(cleaned):\n        return Stage0Result(action=\"bypass\", direct_answer=\"Hello!\", category=\"general\")\n\n    # Tier 0B: Pure math (immediate calc)\n    if RE_MATH_OPERATION.match(cleaned):\n        result = solve_arithmetic(cleaned)\n        if result:\n            return Stage0Result(action=\"bypass\", direct_answer=result, category=\"math\")\n    return Stage0Result(action=\"continue\")\n\n\n# =========================================================================\n# QUALITY VERIFICATION\n# =========================================================================\n\n_DEGENERATE_PATTERNS = [\n    r\"\\bi don'?t know\\b\", r\"\\bi do not know\\b\", r\"\\bi cannot\\b\",\n    r\"\\bi can'?t\\b\", r\"\\bas an ai\\b\", r\"\\bunable to\\b\",\n    r\"\\bno information\\b\", r\"\\bis not provided\\b\", r\"\\bdoes not contain\\b\",\n    r\"\\bcannot answer\\b\", r\"\\bcannot provide\\b\", r\"\\bnot enough information\\b\",\n    r\"\\binsufficient\\b\", r\"\\bsorry\\b\", r\"\\bthe text does not\\b\",\n]\n\n\ndef _has_hedge(text: str) -> bool:\n    low = text.lower()\n    for pat in _DEGENERATE_PATTERNS:\n        if re.search(pat, low):\n            return True\n    return False\n\n\ndef _is_degenerate(text: str) -> bool:\n    words = text.split()\n    if len(words) < 2:\n        return True\n    # Heavy repetition\n    if len(set(words)) / max(len(words), 1) < 0.3:\n        return True\n    return False\n\n\n@dataclass\nclass QCResult:\n    passed: bool\n    reason: str = \"\"\n\n\ndef verify(text: str, task: str = \"\") -> QCResult:\n    \"\"\"Quality check on solver output.\"\"\"\n    text = text.strip()\n    if not text:\n        return QCResult(passed=False, reason=\"empty\")\n    if _has_hedge(text):\n        return QCResult(passed=False, reason=\"hedge_detected\")\n    if _is_degenerate(text):\n        return QCResult(passed=False, reason=\"degenerate\")\n    return QCResult(passed=True)\n\n\n# =========================================================================\n# LOCAL MODEL INFERENCE\n# =========================================================================\n\n_LLM: Optional[object] = None\n_LLM_LOCK = threading.Lock()\n\n\ndef _get_llm(model_path: str, n_gpu_layers: int = -1) -> Optional[object]:\n    \"\"\"Lazy-load the Llama model singleton.\"\"\"\n    global _LLM\n    if _LLM is not None:\n        return _LLM\n    with _LLM_LOCK:\n        if _LLM is not None:\n            return _LLM\n        try:\n            from llama_cpp import Llama\n            logger.info(f\"Loading model from {model_path} (n_gpu_layers={n_gpu_layers})\")\n            _LLM = Llama(\n                model_path=model_path,\n                n_ctx=N_CTX,\n                n_gpu_layers=n_gpu_layers,\n                n_threads=N_THREADS if n_gpu_layers == 0 else 2,\n                verbose=False,\n            )\n            return _LLM\n        except Exception as e:\n            logger.error(f\"Failed to load model: {e}\")\n            return None\n\n\ndef reset_model() -> None:\n    \"\"\"Unload the current model singleton. Must call between model swaps.\"\"\"\n    global _LLM\n    with _LLM_LOCK:\n        if _LLM is not None:\n            del _LLM\n            _LLM = None\n            import gc\n            gc.collect()\n            logger.info(\"Model unloaded\")\n\n\n\n\ndef solve_with_consensus(prompt: str, category: str,\n                         system_prompt: str = \"\",\n                         k: int = 1, max_tokens: int = 256,\n                         model_path: str = DEFAULT_MODEL,\n                         n_gpu_layers: int = -1) -> Optional[Dict[str, Any]]:\n    \"\"\"Single sample (k=1) inference via llama-cpp-python.\"\"\"\n    llm = _get_llm(model_path, n_gpu_layers)\n    if llm is None:\n        return None\n\n    messages = []\n    if system_prompt:\n        messages.append({\"role\": \"system\", \"content\": system_prompt})\n    messages.append({\"role\": \"user\", \"content\": prompt})\n\n    try:\n        resp = llm.create_chat_completion(\n            messages,\n            max_tokens=max_tokens,\n            temperature=0.1,\n            stop=[\"\\n\\n\", \"<|im_end|>\", \"<|eot_id|>\"],\n        )\n        answer = resp[\"choices\"][0][\"message\"][\"content\"].strip()\n        usage = resp.get(\"usage\", {})\n        logger.info(f\"  {category}: {len(answer)} chars, \"\n                     f\"prompt_tok={usage.get('prompt_tokens',0)}, \"\n                     f\"comp_tok={usage.get('completion_tokens',0)}\")\n        return {\n            \"majority_answer\": answer,\n            \"agreement_score\": 1.0,\n            \"usage\": usage,\n            \"finish_reason\": resp[\"choices\"][0].get(\"finish_reason\", \"\"),\n        }\n    except Exception as e:\n        logger.warning(f\"  inference error: {e}\")\n        return None\n\n\n# =========================================================================\n# FUZZY MATCH GRADER\n# =========================================================================\n\ndef _tokenize(text: str) -> List[str]:\n    \"\"\"Simple whitespace + punctuation tokenization.\"\"\"\n    return re.findall(r'\\w+', text.lower())\n\n\ndef fuzzy_match(answer: str, expected: str) -> float:\n    \"\"\"Grade answer against expected using cascade: exact \u2192 substring \u2192 numeric \u2192 token overlap.\"\"\"\n    a = answer.strip().lower()\n    e = expected.strip().lower()\n\n    if not a or not e:\n        return 0.0\n\n    # Exact match\n    if a == e:\n        return 1.0\n\n    # Substring match (expected is contained in answer or vice versa)\n    if e in a or a in e:\n        return 0.95\n\n    # Numeric tolerance\n    nums_a = re.findall(r'-?\\d+\\.?\\d*', a)\n    nums_e = re.findall(r'-?\\d+\\.?\\d*', e)\n    if nums_a and nums_e and len(nums_a) == len(nums_e):\n        try:\n            matches = sum(1 for na, ne in zip(nums_a, nums_e)\n                         if abs(float(na) - float(ne)) / max(abs(float(ne)), 1) < 0.01)\n            if matches == len(nums_a):\n                return 0.9\n        except ValueError:\n            pass\n\n    # Token overlap (Jaccard)\n    tok_a = set(_tokenize(a))\n    tok_e = set(_tokenize(e))\n    if not tok_a or not tok_e:\n        return 0.0\n    overlap = len(tok_a & tok_e)\n    jaccard = overlap / len(tok_a | tok_e)\n    return jaccard\n\n\n# =========================================================================\n# MAIN PIPELINE\n# =========================================================================\n\ndef run_pipeline(prompt: str, model_path: str = DEFAULT_MODEL,\n                 n_gpu_layers: int = -1) -> Dict[str, Any]:\n    \"\"\"Run the full pipeline on one prompt. Returns dict with all results.\"\"\"\n    t_start = time.time()\n    result = {\n        \"answer\": \"\",\n        \"category\": \"\",\n        \"complexity\": 0.0,\n        \"used_api\": False,\n        \"pipeline_stages\": {},\n        \"timing_ms\": {},\n    }\n\n    # \u2500\u2500 Stage 0: Pre-filter \u2500\u2500\n    t0 = time.time()\n    s0 = stage0(prompt)\n    result[\"timing_ms\"][\"stage0\"] = (time.time() - t0) * 1000\n    result[\"pipeline_stages\"][\"stage0\"] = s0.action\n\n    if s0.action == \"bypass\" and s0.direct_answer:\n        result[\"answer\"] = s0.direct_answer\n        result[\"category\"] = s0.category or \"general\"\n        result[\"complexity\"] = 0.0\n        result[\"timing_ms\"][\"total\"] = (time.time() - t_start) * 1000\n        return result\n\n    # \u2500\u2500 Stage 2: Category classifier \u2500\u2500\n    t1 = time.time()\n    category, confidence, scores = classify(prompt)\n    result[\"timing_ms\"][\"stage2\"] = (time.time() - t1) * 1000\n    result[\"category\"] = category\n    result[\"pipeline_stages\"][\"category\"] = category\n    result[\"pipeline_stages\"][\"confidence\"] = confidence\n    result[\"pipeline_stages\"][\"scores\"] = scores\n\n    # \u2500\u2500 Stage 3: Complexity \u2500\u2500\n    t2 = time.time()\n    complexity = score_complexity(prompt, category)\n    result[\"timing_ms\"][\"stage3\"] = (time.time() - t2) * 1000\n    result[\"complexity\"] = complexity\n\n    # \u2500\u2500 Stage 4: Decision \u2500\u2500\n    needs_api = True\n    if complexity < COMPLEXITY_SIMPLE_MAX and category in DETERMINISTIC_CATEGORIES and category not in NAKED_CATEGORIES:\n        det_cat = DET_CATEGORY_MAP.get(category, \"other_complex\")\n        solver_fn = DET_SOLVER_FNS.get(det_cat)\n        if solver_fn:\n            t3 = time.time()\n            try:\n                ans = solver_fn(prompt, det_cat)\n                result[\"timing_ms\"][\"deterministic\"] = (time.time() - t3) * 1000\n                if ans:\n                    result[\"answer\"] = ans\n                    result[\"pipeline_stages\"][\"solver\"] = f\"deterministic:{det_cat}\"\n                    needs_api = False\n            except Exception as e:\n                logger.warning(f\"  Deterministic solver error: {e}\")\n\n    # \u2500\u2500 API path \u2500\u2500\n    if needs_api:\n        result[\"pipeline_stages\"][\"solver\"] = \"local_llm\"\n\n        # Build system prompt\n        sys_prompt = \"\"\n        if category not in NAKED_CATEGORIES:\n            ner_example = NER_ONE_SHOT_EXAMPLE if category == \"ner\" else None\n            sys_prompt = build_system_prompt(category, complexity, custom_instructions=ner_example or \"\")\n            result[\"pipeline_stages\"][\"prompt_tier\"] = (\n                \"low\" if complexity < 0.3 else (\"medium\" if complexity < 0.6 else \"high\")\n            )\n\n        t4 = time.time()\n        result_obj = solve_with_consensus(\n            prompt=prompt,\n            category=category,\n            system_prompt=sys_prompt,\n            k=1,\n            max_tokens=get_max_tokens(category, complexity),\n            model_path=model_path,\n            n_gpu_layers=n_gpu_layers,\n        )\n        result[\"timing_ms\"][\"local_llm\"] = (time.time() - t4) * 1000\n\n        if result_obj and result_obj.get(\"majority_answer\"):\n            result[\"answer\"] = result_obj[\"majority_answer\"]\n            result[\"pipeline_stages\"][\"agreement\"] = result_obj.get(\"agreement_score\", 0)\n            if result_obj.get(\"usage\"):\n                result[\"pipeline_stages\"][\"prompt_tokens\"] = result_obj[\"usage\"].get(\"prompt_tokens\", 0)\n                result[\"pipeline_stages\"][\"completion_tokens\"] = result_obj[\"usage\"].get(\"completion_tokens\", 0)\n\n    # \u2500\u2500 QC Gate \u2500\u2500\n    if result.get(\"answer\"):\n        qc = verify(result[\"answer\"], task=category)\n        result[\"pipeline_stages\"][\"qc_pass\"] = qc.passed\n        if not qc.passed:\n            result[\"pipeline_stages\"][\"qc_reason\"] = qc.reason\n            result[\"answer\"] = \"\"  # Discard failed answer\n\n    result[\"timing_ms\"][\"total\"] = (time.time() - t_start) * 1000\n    return result\n\n\ndef grade_answer(answer: str, expected: str) -> Dict[str, Any]:\n    \"\"\"Grade a single answer against expected.\"\"\"\n    score = fuzzy_match(answer, expected)\n    return {\n        \"score\": score,\n        \"passed\": score >= 0.5,\n        \"answer\": answer,\n        \"expected\": expected,\n    }\n\n\n# =========================================================================\n# EVAL RUNNER\n# =========================================================================\n\ndef load_eval_data(path: str) -> List[Dict[str, Any]]:\n    \"\"\"Load eval data from JSON file (supports training-v1.json format).\"\"\"\n    with open(path) as f:\n        data = json.load(f)\n    # Handle dict with 'questions' key (eval_clean_val.json format)\n    if isinstance(data, dict) and \"questions\" in data:\n        data = data[\"questions\"]\n    logger.info(f\"Loaded {len(data)} questions from {path}\")\n    return data\n\n\ndef run_eval(eval_path: str, model_path: str = DEFAULT_MODEL,\n             n_gpu_layers: int = -1, max_questions: int = 0,\n             results_dir: str = \"results\",\n             model_label: str = \"\") -> Tuple[List[Dict[str, Any]], Dict[str, Any]]:\n    \"\"\"Run full pipeline on eval dataset. Returns per-question results.\"\"\"\n    os.makedirs(results_dir, exist_ok=True)\n    questions = load_eval_data(eval_path)\n\n    if max_questions > 0:\n        questions = questions[:max_questions]\n        logger.info(f\"Limited to {max_questions} questions\")\n\n    # Warm up model\n    logger.info(\"Warming up model...\")\n    t_warm = time.time()\n    llm = _get_llm(model_path, n_gpu_layers)\n    if llm:\n        try:\n            llm.create_chat_completion(\n                [{\"role\": \"user\", \"content\": \"Hello\"}], max_tokens=5\n            )\n        except Exception:\n            pass\n        logger.info(f\"Model warmup: {time.time() - t_warm:.1f}s\")\n    else:\n        logger.warning(\"Model not loaded \u2014 pipeline will use deterministic solvers only\")\n\n    # Stats tracking\n    per_category = {}\n    total_time = 0.0\n    results = []\n\n    for idx, q in enumerate(questions):\n        prompt = q.get(\"prompt\", q.get(\"text\", \"\"))\n        expected = q.get(\"expected_answer\", q.get(\"label\", q.get(\"answer\", \"\")))\n        category = q.get(\"category_label\", q.get(\"category\", \"\"))\n        task_id = q.get(\"task_id\", q.get(\"id\", f\"q_{idx}\"))\n\n        logger.info(f\"[{idx + 1}/{len(questions)}] {task_id} ({category}): {prompt[:60]}...\")\n\n        t_q = time.time()\n        pipe_result = run_pipeline(prompt, model_path, n_gpu_layers)\n        q_time = time.time() - t_q\n\n        # Grade\n        grading = grade_answer(pipe_result.get(\"answer\", \"\"), expected)\n\n        # Collect\n        result_row = {\n            \"idx\": idx,\n            \"task_id\": task_id,\n            \"model_label\": model_label,\n            \"category\": pipe_result.get(\"category\", category),\n            \"prompt\": prompt[:120],\n            \"expected\": expected[:120],\n            \"answer\": pipe_result.get(\"answer\", \"\")[:120],\n            \"score\": round(grading[\"score\"], 4),\n            \"passed\": grading[\"passed\"],\n            \"total_time_s\": round(q_time, 3),\n            \"stage0_ms\": round(pipe_result[\"timing_ms\"].get(\"stage0\", 0), 1),\n            \"stage2_ms\": round(pipe_result[\"timing_ms\"].get(\"stage2\", 0), 1),\n            \"stage3_ms\": round(pipe_result[\"timing_ms\"].get(\"stage3\", 0), 1),\n            \"det_ms\": round(pipe_result[\"timing_ms\"].get(\"deterministic\", 0), 1),\n            \"llm_ms\": round(pipe_result[\"timing_ms\"].get(\"local_llm\", 0), 1),\n            \"complexity\": round(pipe_result.get(\"complexity\", 0), 3),\n            \"confidence\": round(pipe_result.get(\"pipeline_stages\", {}).get(\"confidence\", 0), 3),\n            \"raw_scores\": pipe_result.get(\"pipeline_stages\", {}).get(\"scores\", {}),\n            \"max_tokens\": get_max_tokens(pipe_result.get(\"category\", category) or \"factual\", pipe_result.get(\"complexity\", 0.5)),\n            \"qc_pass\": pipe_result.get(\"pipeline_stages\", {}).get(\"qc_pass\", False),\n            \"solver\": pipe_result.get(\"pipeline_stages\", {}).get(\"solver\", \"\"),\n            \"prompt_tokens\": pipe_result.get(\"pipeline_stages\", {}).get(\"prompt_tokens\", 0),\n            \"completion_tokens\": pipe_result.get(\"pipeline_stages\", {}).get(\"completion_tokens\", 0),\n        }\n        results.append(result_row)\n\n        # Per-category stats\n        cat_name = pipe_result.get(\"category\", category) or \"general\"\n        if cat_name not in per_category:\n            per_category[cat_name] = {\"count\": 0, \"passed\": 0, \"time\": 0.0}\n        per_category[cat_name][\"count\"] += 1\n        per_category[cat_name][\"passed\"] += 1 if grading[\"passed\"] else 0\n        per_category[cat_name][\"time\"] += q_time\n        total_time += q_time\n\n        if (idx + 1) % 100 == 0:\n            passed_sofar = sum(1 for r in results if r[\"passed\"])\n            logger.info(f\"  --- [{idx+1}/{len(questions)}] accuracy so far: {passed_sofar}/{idx+1} = {passed_sofar/(idx+1)*100:.1f}%\")\n\n    # Compute summary\n    summary = {\n        \"eval_path\": eval_path,\n        \"model_path\": model_path,\n        \"n_gpu_layers\": n_gpu_layers,\n        \"total_questions\": len(results),\n        \"total_time_s\": round(total_time, 1),\n        \"avg_time_per_q_s\": round(total_time / max(len(results), 1), 3),\n        \"questions_per_min\": round(len(results) / max(total_time / 60, 0.01), 1),\n        \"overall_accuracy\": round(\n            sum(1 for r in results if r[\"passed\"]) / max(len(results), 1) * 100, 1\n        ),\n        \"passed\": sum(1 for r in results if r[\"passed\"]),\n        \"failed\": sum(1 for r in results if not r[\"passed\"]),\n        \"timestamp\": datetime.now().isoformat(),\n        \"per_category\": {},\n    }\n    for cat, stats in sorted(per_category.items()):\n        accuracy = round(stats[\"passed\"] / max(stats[\"count\"], 1) * 100, 1)\n        avg_time = round(stats[\"time\"] / max(stats[\"count\"], 1), 3)\n        summary[\"per_category\"][cat] = {\n            \"count\": stats[\"count\"], \"passed\": stats[\"passed\"],\n            \"accuracy_pct\": accuracy, \"avg_time_s\": avg_time,\n        }\n\n    return results, summary\n\n\n# =========================================================================\n# OUTPUT HELPERS\n# =========================================================================\n\ndef _col_letter(n: int) -> str:\n    \"\"\"Convert 1-indexed column number to Excel column letter (A, B, ... Z, AA, AB).\"\"\"\n    result = \"\"\n    while n > 0:\n        n -= 1\n        result = chr(65 + n % 26) + result\n        n //= 26\n    return result\n\ndef write_xlsx(results: List[Dict], summary: Dict, output_path: str) -> str:\n    \"\"\"Write results to xlsx matching RunLogger format: Run Meta, Questions, Raw Scores sheets.\n    \n    Adds Score, Passed, and Expected Answer columns to the Questions sheet.\n    \"\"\"\n    try:\n        import openpyxl\n        from openpyxl.styles import Font, PatternFill, Alignment, Border, Side\n    except ImportError:\n        logger.warning(\"openpyxl not available \u2014 skipping xlsx\")\n        return \"\"\n\n    os.makedirs(os.path.dirname(output_path) or \".\", exist_ok=True)\n    wb = openpyxl.Workbook()\n\n    header_fill = PatternFill(start_color=\"4472C4\", end_color=\"4472C4\", fill_type=\"solid\")\n    header_font = Font(bold=True, color=\"FFFFFF\")\n    bold = Font(bold=True)\n    thin_border = Border(\n        left=Side(style=\"thin\"), right=Side(style=\"thin\"),\n        top=Side(style=\"thin\"), bottom=Side(style=\"thin\"),\n    )\n\n    # \u2500\u2500 Sheet 1: Run Meta \u2500\u2500\n    meta = wb.active\n    meta.title = \"Run Meta\"\n    meta.column_dimensions[\"A\"].width = 30\n    meta.column_dimensions[\"B\"].width = 60\n\n    meta_rows = [\n        (\"Run Number\", summary.get(\"run_number\", 1)),\n        (\"Run Timestamp\", summary.get(\"timestamp\", \"\")),\n        (\"Pipeline Version\", \"colab-pipeline\"),\n        (\"Total Questions\", summary.get(\"total_questions\", len(results))),\n        (\"Eval Source\", os.path.basename(summary.get(\"eval_path\", \"\"))),\n        (\"Model Path\", summary.get(\"model_path\", \"\")),\n        (\"Model Label\", summary.get(\"model_label\", \"\")),\n        (\"Fireworks Model\", \"\"),\n        (\"Fireworks Key Set\", \"False\"),\n        (\"N GPU Layers\", str(summary.get(\"n_gpu_layers\", -1))),\n        (\"N Context\", \"2048\"),\n        (\"N Threads\", \"4\"),\n        (\"Total Elapsed (s)\", f\"{summary.get('total_time_s', 0):.1f}\"),\n        (\"Questions Logged\", str(len(results))),\n        (\"Overall Accuracy\", f\"{summary.get('overall_accuracy', 0)}%\"),\n        (\"Avg Time Per Q (s)\", f\"{summary.get('avg_time_per_q_s', 0):.3f}\"),\n    ]\n    for i, (k, v) in enumerate(meta_rows, 1):\n        c1 = meta.cell(row=i, column=1, value=k); c1.font = bold; c1.border = thin_border\n        c2 = meta.cell(row=i, column=2, value=v); c2.border = thin_border\n\n    # \u2500\u2500 Sheet 2: Questions \u2500\u2500\n    qs = wb.create_sheet(\"Questions\")\n    cols = [\n        \"Run\", \"Task ID\", \"Model\",\n        \"Input Prompt\", \"Expected Answer\", \"Final Answer\", \"Score\", \"Passed\",\n        \"Pre-Filter Action\", \"Pre-Filter Answer\", \"Pre-Filter (ms)\",\n        \"Category\", \"Confidence\",\n        \"Raw Scores\",\n        \"Complexity\", \"Complexity (ms)\",\n        \"Solver Name\", \"Max Tokens\",\n        \"Prompt Version\",\n        \"QC Pass\",\n        \"Det (ms)\",\n        \"Local LLM (ms)\", \"Prompt Tokens\", \"Completion Tokens\", \"Total Tokens\",\n        \"Error\",\n        \"Total (ms)\",\n    ]\n    for ci, h in enumerate(cols, 1):\n        cell = qs.cell(row=1, column=ci, value=h)\n        cell.font = header_font; cell.fill = header_fill\n        cell.alignment = Alignment(wrap_text=True); cell.border = thin_border\n\n    for ri, r in enumerate(results, 2):\n        vals = [\n            1, r.get(\"task_id\", \"\"), r.get(\"model_label\", \"\"),\n            r.get(\"prompt\", \"\"), r.get(\"expected\", \"\"), r.get(\"answer\", \"\"),\n            r.get(\"score\", 0), r.get(\"passed\", False),\n            \"\", \"\", r.get(\"stage0_ms\", 0),\n            r.get(\"category\", \"\"), r.get(\"confidence\", 0),\n            \"\",\n            r.get(\"complexity\", 0), r.get(\"stage3_ms\", 0),\n            r.get(\"solver\", \"\"), r.get(\"max_tokens\", 0),\n            \"\",\n            r.get(\"qc_pass\", \"\"),\n            r.get(\"det_ms\", 0),\n            r.get(\"llm_ms\", 0), r.get(\"prompt_tokens\", 0), r.get(\"completion_tokens\", 0),\n            r.get(\"prompt_tokens\", 0) + r.get(\"completion_tokens\", 0),\n            r.get(\"error\", \"\"),\n            round(r.get(\"total_time_s\", 0) * 1000, 1),\n        ]\n        for ci, v in enumerate(vals, 1):\n            cell = qs.cell(row=ri, column=ci, value=v)\n            cell.border = thin_border\n            if isinstance(v, str) and len(v) > 300:\n                cell.alignment = Alignment(wrap_text=True)\n\n    qs.auto_filter.ref = f\"A1:{_col_letter(len(cols))}{len(results) + 1}\"\n    qs.freeze_panes = \"C2\"\n\n    # \u2500\u2500 Sheet 3: Raw Scores \u2500\u2500\n    raw = wb.create_sheet(\"Raw Scores\")\n    raw_cols = [\"Run\", \"Task ID\", \"Category\", \"Confidence\",\n                \"code_debug\", \"code_gen\", \"factual\", \"logic\",\n                \"math\", \"ner\", \"sentiment\", \"summarization\"]\n    for ci, h in enumerate(raw_cols, 1):\n        cell = raw.cell(row=1, column=ci, value=h)\n        cell.font = header_font; cell.fill = header_fill; cell.border = thin_border\n\n    for ri, r in enumerate(results, 2):\n        raw_scores = r.get(\"raw_scores\", {}) or {}\n        if isinstance(raw_scores, str):\n            try: raw_scores = json.loads(raw_scores)\n            except: raw_scores = {}\n        vals = [\n            1, r.get(\"task_id\", \"\"), r.get(\"category\", \"\"), r.get(\"confidence\", 0),\n            round(raw_scores.get(\"code_debug\", 0), 1),\n            round(raw_scores.get(\"code_gen\", 0), 1),\n            round(raw_scores.get(\"factual\", 0), 1),\n            round(raw_scores.get(\"logic\", 0), 1),\n            round(raw_scores.get(\"math\", 0), 1),\n            round(raw_scores.get(\"ner\", 0), 1),\n            round(raw_scores.get(\"sentiment\", 0), 1),\n            round(raw_scores.get(\"summarization\", 0), 1),\n        ]\n        for ci, v in enumerate(vals, 1):\n            cell = raw.cell(row=ri, column=ci, value=v); cell.border = thin_border\n\n    raw.auto_filter.ref = f\"A1:L{len(results) + 1}\"\n    raw.freeze_panes = \"C2\"\n\n    # \u2500\u2500 Sheet 4: Per-Category Per-Model Summary (comparison) \u2500\u2500\n    if summary.get(\"per_category\"):\n        comp = wb.create_sheet(\"Per-Category Summary\")\n        comp_cols = [\"Category\", \"Count\", \"Passed\", \"Accuracy %\", \"Avg Time (s)\"]\n        for ci, h in enumerate(comp_cols, 1):\n            cell = comp.cell(row=1, column=ci, value=h)\n            cell.font = header_font; cell.fill = header_fill; cell.border = thin_border\n        for ri, (cat, s) in enumerate(sorted(summary.get(\"per_category\", {}).items()), 2):\n            vals_cat = [cat, s[\"count\"], s[\"passed\"], s[\"accuracy_pct\"], s[\"avg_time_s\"]]\n            for ci, v in enumerate(vals_cat, 1):\n                cell = comp.cell(row=ri, column=ci, value=v); cell.border = thin_border\n\n    wb.save(output_path)\n    logger.info(f\"xlsx written to {output_path}\")\n    return output_path\n\n\ndef write_results(results: List[Dict], summary: Dict,\n                  results_dir: str = \"results\") -> str:\n    \"\"\"Write results to JSON and summary text files.\"\"\"\n    os.makedirs(results_dir, exist_ok=True)\n    timestamp = datetime.now().strftime(\"%Y%m%d_%H%M%S\")\n    base = os.path.join(results_dir, f\"colab_run_{timestamp}\")\n\n    # JSON\n    json_path = f\"{base}.json\"\n    with open(json_path, \"w\") as f:\n        json.dump({\"summary\": summary, \"results\": results}, f, indent=2)\n    logger.info(f\"Results written to {json_path}\")\n\n    # CSV\n    csv_path = f\"{base}.csv\"\n    if results:\n        with open(csv_path, \"w\", newline=\"\") as f:\n            writer = csv.DictWriter(f, fieldnames=results[0].keys())\n            writer.writeheader()\n            writer.writerows(results)\n        logger.info(f\"CSV written to {csv_path}\")\n\n    # Text summary\n    txt_path = f\"{base}_summary.txt\"\n    with open(txt_path, \"w\") as f:\n        f.write(\"=\" * 60 + \"\\n\")\n        f.write(\"COLAB PIPELINE EVAL SUMMARY\\n\")\n        f.write(f\"Run: {summary['timestamp']}\\n\")\n        f.write(\"=\" * 60 + \"\\n\\n\")\n        f.write(f\"Eval file:      {summary['eval_path']}\\n\")\n        f.write(f\"Model:          {summary['model_path']}\\n\")\n        f.write(f\"GPU layers:     {summary['n_gpu_layers']}\\n\")\n        f.write(f\"Total questions: {summary['total_questions']}\\n\")\n        f.write(f\"Passed:         {summary['passed']}\\n\")\n        f.write(f\"Failed:         {summary['failed']}\\n\")\n        f.write(f\"Overall accuracy: {summary['overall_accuracy']}%\\n\\n\")\n        f.write(f\"Total time:     {summary['total_time_s']}s\\n\")\n        f.write(f\"Avg per Q:      {summary['avg_time_per_q_s']}s\\n\")\n        f.write(f\"Questions/min:  {summary['questions_per_min']}\\n\\n\")\n        f.write(\"Per-Category Breakdown:\\n\")\n        f.write(\"-\" * 60 + \"\\n\")\n        f.write(f\"{'Category':<16} {'Count':>6} {'Passed':>6} {'Accuracy':>10} {'Avg Time':>10}\\n\")\n        f.write(\"-\" * 60 + \"\\n\")\n        for cat, s in sorted(summary[\"per_category\"].items()):\n            f.write(f\"{cat:<16} {s['count']:>6} {s['passed']:>6} \"\n                    f\"{s['accuracy_pct']:>8.1f}% {s['avg_time_s']:>8.3f}s\\n\")\n        f.write(\"=\" * 60 + \"\\n\")\n    logger.info(f\"Summary written to {txt_path}\")\n\n    return base\n\n\ndef print_summary(summary: Dict):\n    \"\"\"Print summary to stdout.\"\"\"\n    print()\n    print(\"=\" * 60)\n    print(\"COLAB PIPELINE RESULTS\")\n    print(\"=\" * 60)\n    print(f\"  Questions:    {summary['total_questions']}\")\n    print(f\"  Passed:       {summary['passed']}\")\n    print(f\"  Accuracy:     {summary['overall_accuracy']}%\")\n    print(f\"  Total time:   {summary['total_time_s']:.1f}s\")\n    print(f\"  Avg per Q:    {summary['avg_time_per_q_s']:.3f}s\")\n    print(f\"  Q/min:        {summary['questions_per_min']:.1f}\")\n    print(\"-\" * 60)\n    print(f\"  {'Category':<16} {'Count':>6} {'Acc%':>6} {'Avg/s':>7}\")\n    print(\"-\" * 60)\n    for cat, s in sorted(summary[\"per_category\"].items()):\n        print(f\"  {cat:<16} {s['count']:>6} {s['accuracy_pct']:>5.1f}% {s['avg_time_s']:>6.3f}\")\n    print(\"=\" * 60)\n\n\ndef main():\n    parser = argparse.ArgumentParser(description=\"Colab pipeline evaluator\")\n    parser.add_argument(\"--eval\", required=True, help=\"Path to eval JSON file\")\n    parser.add_argument(\"--model\", default=DEFAULT_MODEL, help=\"Path to GGUF model\")\n    parser.add_argument(\"--gpu-layers\", type=int, default=N_GPU_LAYERS, help=\"GPU layers (-1=all)\")\n    parser.add_argument(\"--max\", type=int, default=0, help=\"Max questions (0=all)\")\n    parser.add_argument(\"--results-dir\", default=\"results\", help=\"Output directory\")\n    parser.add_argument(\"--model-label\", default=\"\", help=\"Label for this model (e.g. qwen-1.5b)\")\n    args = parser.parse_args()\n\n    results, summary = run_eval(\n        eval_path=args.eval,\n        model_path=args.model,\n        n_gpu_layers=args.gpu_layers,\n        max_questions=args.max,\n        results_dir=args.results_dir,\n        model_label=args.model_label,\n    )\n\n    base_path = write_results(results, summary, args.results_dir)\n\n    # Write xlsx\n    xlsx_path = os.path.join(args.results_dir, f\"{os.path.basename(base_path)}.xlsx\")\n    write_xlsx(results, summary, xlsx_path)\n\n    print_summary(summary)\n    print(f\"\\nResults saved: {base_path}.*\")\n\n\nif __name__ == \"__main__\":\n    main()\n"
with open("colab_pipeline.py", "w") as f:
    f.write(PIPELINE_SCRIPT)
print(f"Written: {len(PIPELINE_SCRIPT)} bytes")


## Step 3 — Download models (~3.9 GB, one-time cache)

In [ ]:
# ── 5. Download all 4 models ──
for m in MODELS:
    if not os.path.exists(m['path']):
        print(f"Downloading {m['label']}...")
        !wget -q --show-progress "{m['url']}" -O "{m['path']}"
        print(f"  {os.path.getsize(m['path'])/1e6:.0f} MB")
    else:
        print(f"{m['label']} cached ({os.path.getsize(m['path'])/1e6:.0f} MB)")


## Step 4 — Run all 4 models on all 6 datasets

**Takes ~2 hours.** Stay on this tab. If disconnected, re-run from here.


In [ ]:
# ── 6. RUN EVERYTHING ──
os.makedirs("results", exist_ok=True)
all_summaries = []

for mi, model in enumerate(MODELS):
    label = model['label']; path = model['path']
    print(f"\n{'='*70}\nMODEL {mi+1}/4: {label}\n{'='*70}")
    for fi, ef in enumerate(eval_files):
        base = os.path.splitext(os.path.basename(ef))[0]
        print(f"  [{fi+1}/{len(eval_files)}] {base}...")
        !python3 colab_pipeline.py --eval "{ef}" --model "{path}" --gpu-layers -1 --model-label "{label}" --results-dir results 2>&1 | tail -1
    for ef in eval_files:
        base = os.path.splitext(os.path.basename(ef))[0]
        for f in sorted(glob.glob(f"results/*{label}*{base}*.json")):
            with open(f) as fh: r = json.load(fh); all_summaries.append(r.get('summary', r))
    print(f"  {label} DONE")

print(f"\nALL 4 MODELS COMPLETE — {len(glob.glob('results/*.xlsx'))} xlsx files")


## Comparison table

In [ ]:
# ── 7. Comparison ──
if not all_summaries:
    for model in MODELS:
        for ef in eval_files:
            base = os.path.splitext(os.path.basename(ef))[0]
            for f in sorted(glob.glob(f"results/*{model['label']}*{base}*.json")):
                with open(f) as fh: all_summaries.append(json.load(fh).get('summary', {}))

CATS = ["code_debug","code_gen","factual","logic","math","ner","sentiment","summarization"]
mlabs = sorted(set(s.get('model_label','?') for s in all_summaries))
print("="*90 + "\nCROSS-MODEL ACCURACY (avg across 6 datasets)\n" + "="*90)
hdr = f"{'Category':<16}"
for m in mlabs: hdr += f" {m:>16}"
print(hdr + "  Winner")
print("-"*90)
for cat in CATS:
    row, accs = f"{cat:<16}", []
    for m in mlabs:
        v = [s.get('per_category',{}).get(cat,{}).get('accuracy_pct',0) for s in all_summaries if s.get('model_label')==m and s.get('per_category',{}).get(cat)]
        a = sum(v)/len(v) if v else 0.0; accs.append(a)
    mx = max(accs)
    for i, a in enumerate(accs): row += f"  {a:>13.1f}%{'*' if a==mx and mx>0 else ' '}"
    row += f"  {mlabs[accs.index(mx)]}"
    print(row)
print("="*90)


## Step 5 — Download results

**Click the button below to get all 24 xlsx files as a single zip.**

In [ ]:
# ── 8. Download results (ONE BUTTON) ──
from google.colab import files
import zipfile, io

result_files = sorted(glob.glob("results/*"))
print(f"Packing {len(result_files)} files ({sum(os.path.getsize(f) for f in result_files)/1024:.0f} KB)...")

zip_buffer = io.BytesIO()
with zipfile.ZipFile(zip_buffer, 'w', zipfile.ZIP_DEFLATED) as zf:
    for f in result_files:
        zf.write(f, arcname=os.path.basename(f))
zip_buffer.seek(0)
with open("results/all_results.zip", "wb") as f:
    f.write(zip_buffer.read())

files.download("results/all_results.zip")
print("Done! all_results.zip downloaded to your machine.")
